# Step 1.2 — Lemmatization

**Goal:** Reduce all words in `text_of_law` to their base form (lemma) so that morphological variants of the same word are treated as identical tokens.

**Why lemmatization first?**  
Without lemmatization, `Kanton`, `Kantons`, `Kantone` and `kantonalen` are counted as four distinct tokens. After lemmatization, all map to `Kanton`. This is critical for the next step (Step 1.3), where we use document frequency to identify and remove legal boilerplate vocabulary: lemmatization ensures that the true frequency of a concept is visible, rather than being diluted across its inflected forms.

**Tool:** We use `spaCy` with the `de_core_news_sm` German language model. In addition to lemmatization, we use spaCy to remove standard German stopwords and punctuation, which are uninformative for clustering regardless of legal domain.

## 1. Setup

In [2]:
import pandas as pd
import spacy
from tqdm import tqdm

tqdm.pandas()

RAW_DATA_PATH = "../data/raw/sgbs_raw.csv"
LEMMA_DATA_PATH = "../data/processed/sgbs_lemmatized.csv"

import os
os.makedirs("../data/processed", exist_ok=True)

# Load German spaCy model
nlp = spacy.load("de_core_news_sm")
print(f"spaCy model loaded: {nlp.meta['name']} (v{nlp.meta['version']})")

spaCy model loaded: core_news_sm (v3.8.0)


## 2. Load Raw Data

In [3]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Records loaded: {len(df)}")
df.head(2)

Records loaded: 733


,title_de,keywords_de,text_of_law,systematic_number,category_name,category_id
0,Entfernung der Spitalliste aus der Gesetzessam...,NaN,Spitäler\n\n\n 330.500\n\n\n Entfernung der ...,330.500,Anderes,5.0
1,Staatsvertrag zwischen den Kantonen Basel-Stad...,NaN,Gesundheitsversorgung: Staatsvertrag BL + BS |...,333.200,Interkantonale Vereinbarung,2.0


## 3. Lemmatization Function

For each document we:
1. Parse the text with spaCy
2. Remove tokens that are stopwords, punctuation, whitespace, or numbers
3. Convert each remaining token to its lemma (lowercase)

We use `nlp.pipe()` for batch processing, which is significantly faster than processing documents one by one.

In [4]:
def lemmatize_text(text):
    """Tokenize, filter, and lemmatize a German legal text."""
    doc = nlp(text)
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop        # remove German stopwords
        and not token.is_punct      # remove punctuation
        and not token.is_space      # remove whitespace tokens
        and not token.like_num      # remove numbers
        and len(token.lemma_) > 2   # remove very short tokens
    ]
    return " ".join(lemmas)

In [5]:
# Process in batches for efficiency
# nlp.max_length may need to be increased for long documents
nlp.max_length = 2_000_000

print("Lemmatizing documents (this may take a few minutes)...")

texts = df["text_of_law"].tolist()
lemmatized = []

for doc in tqdm(nlp.pipe(texts, batch_size=32, disable=["ner", "parser"]), total=len(texts)):
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and not token.like_num
        and len(token.lemma_) > 2
    ]
    lemmatized.append(" ".join(lemmas))

df["text_lemmatized"] = lemmatized
print("Done.")

Lemmatizing documents (this may take a few minutes)...


100%|████████████████████████████████████████████████████████████████████████████████| 733/733 [01:48<00:00,  6.74it/s]

Done.


## 4. Inspect Results

In [6]:
# Compare original vs lemmatized for one document
idx = 0
print("=== ORIGINAL (first 300 chars) ===")
print(df["text_of_law"].iloc[idx][:300])
print("\n=== LEMMATIZED (first 300 chars) ===")
print(df["text_lemmatized"].iloc[idx][:300])

=== ORIGINAL (first 300 chars) ===
Spitäler


  330.500


  Entfernung der Spitalliste aus der Gesetzessammlung



  Vom 27. November 2023 (Stand 1. Januar 2024)



   
    Bisher wurde die Spitalliste in den Kantonen Basel-Landschaft und Basel-Stadt in die Gesetzessammlung aufgenommen. Da die Spitalliste die wesentlichen Inhalte der

=== LEMMATIZED (first 300 chars) ===
spitäler entfernung spitalliste gesetzessammlung november stand januar spitalliste kanton basel-landschaft basel-stadt gesetzessammlung aufnehmen spitalliste wesentlich inhalt leistungsauftrag öffentlich öffentlich register gleichzustellen angesichts rechtsnatur spitalliste zukünftig aufnahme gesetz


In [7]:
# Token count before and after
df["token_count_original"] = df["text_of_law"].str.split().str.len()
df["token_count_lemmatized"] = df["text_lemmatized"].str.split().str.len()

print(f"Avg tokens before lemmatization: {df['token_count_original'].mean():.0f}")
print(f"Avg tokens after lemmatization:  {df['token_count_lemmatized'].mean():.0f}")
print(f"Avg reduction: {(1 - df['token_count_lemmatized'] / df['token_count_original']).mean():.1%}")

Avg tokens before lemmatization: 1423
Avg tokens after lemmatization:  675
Avg reduction: 52.2%


## 5. Persist

In [8]:
df.drop(columns=["token_count_original", "token_count_lemmatized"]).to_csv(LEMMA_DATA_PATH, index=False)
print(f"Saved {len(df)} records to {LEMMA_DATA_PATH}")

Saved 733 records to ../data/processed/sgbs_lemmatized.csv


## 6. Summary

**What we did:**
- Loaded 733 active cantonal laws from `data/raw/sgbs_raw.csv`
- Applied spaCy `de_core_news_sm` to lemmatize all texts
- Removed German stopwords, punctuation, numbers, and tokens shorter than 3 characters
- Added column `text_lemmatized` alongside the original `text_of_law`
- Persisted as `data/processed/sgbs_lemmatized.csv`

**Next step:** `01_3_legal_stopwords.ipynb` — frequency analysis on lemmatized tokens to identify and remove legal boilerplate vocabulary.